# NBA Game Data — EDA

Read raw game data from S3 and explore.

In [13]:
import json
import boto3
import pandas as pd

S3_BUCKET = "prediction-markets-data"
s3 = boto3.client("s3")

In [14]:
# List what's in the bucket
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="nba/")
for obj in resp.get("Contents", []):
    print(f"{obj['Key']}  ({obj['Size'] / 1024:.0f} KB)")

nba/games/season_2024-25.json  (1219 KB)


In [15]:
# Load a season from S3
SEASON = "2024-25"

obj = s3.get_object(Bucket=S3_BUCKET, Key=f"nba/games/season_{SEASON}.json")
raw = json.loads(obj["Body"].read())
df = pd.DataFrame(raw)
print(f"{len(df)} rows, {df['GAME_ID'].nunique()} games")
df.head()

2802 rows, 1401 games


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,42024,1610612760,OKC,Oklahoma City Thunder,0042400407,2025-06-22,OKC vs. IND,W,240,103,...,0.710,13,27,40,20,14,8,7,23,12.0
1,42024,1610612754,IND,Indiana Pacers,0042400407,2025-06-22,IND @ OKC,L,240,91,...,0.759,12,33,45,17,6,4,21,24,-12.0
2,42024,1610612760,OKC,Oklahoma City Thunder,0042400406,2025-06-19,OKC @ IND,L,240,91,...,0.808,4,37,41,14,4,4,21,20,-17.0
3,42024,1610612754,IND,Indiana Pacers,0042400406,2025-06-19,IND vs. OKC,W,240,108,...,0.680,11,35,46,23,16,5,10,17,17.0
4,42024,1610612760,OKC,Oklahoma City Thunder,0042400405,2025-06-16,OKC vs. IND,W,239,120,...,0.813,19,26,45,24,15,12,11,24,11.0


In [16]:
# Derive game type from GAME_ID prefix (3rd digit)
# 1 = Preseason, 2 = Regular Season, 3 = All-Star, 4 = Playoffs
game_type_map = {"1": "Preseason", "2": "Regular Season", "3": "All-Star", "4": "Playoffs"}
df["GAME_TYPE"] = df["GAME_ID"].astype(str).str[2].map(game_type_map)
df["GAME_TYPE"].value_counts()

GAME_TYPE
Regular Season    2460
Playoffs           168
Preseason          146
All-Star            14
Name: count, dtype: int64

In [17]:
df.shape

(2802, 29)

In [18]:
df.describe()

,TEAM_ID,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
count,2.802000e+03,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,...,2796.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000,2802.000000
mean,1.608314e+09,240.614204,112.940043,41.302641,88.737330,0.466500,13.399714,37.405068,0.357847,16.935046,...,0.777370,11.086010,32.889008,43.975018,26.225910,8.238401,4.885439,13.606353,18.767666,0.001285
std,6.082055e+07,13.355812,13.874430,5.628362,8.359571,0.055139,3.997157,7.268632,0.081508,5.888169,...,0.101171,3.961537,5.639170,6.994675,5.479246,3.130663,2.477764,4.119892,4.428485,16.202305
min,1.502000e+04,39.000000,14.000000,6.000000,19.000000,0.293000,0.000000,8.000000,0.000000,0.000000,...,0.250000,0.000000,3.000000,3.000000,4.000000,0.000000,0.000000,1.000000,0.000000,-59.000000
25%,1.610613e+09,239.000000,104.000000,38.000000,84.000000,0.429000,11.000000,32.000000,0.303000,13.000000,...,0.714000,8.000000,29.000000,39.000000,23.000000,6.000000,3.000000,11.000000,16.000000,-10.000000
50%,1.610613e+09,240.000000,113.000000,41.000000,89.000000,0.466000,13.000000,37.000000,0.357000,17.000000,...,0.783000,11.000000,33.000000,44.000000,26.000000,8.000000,5.000000,13.000000,19.000000,0.000000
75%,1.610613e+09,241.000000,122.000000,45.000000,94.000000,0.505000,16.000000,42.000000,0.412000,21.000000,...,0.846000,14.000000,37.000000,48.000000,30.000000,10.000000,6.000000,16.000000,22.000000,10.000000
max,1.610617e+09,292.000000,162.000000,60.000000,119.000000,0.783000,29.000000,63.000000,0.680000,43.000000,...,1.000000,28.000000,52.000000,73.000000,48.000000,22.000000,18.000000,32.000000,37.000000,59.000000


In [19]:
# Win/loss distribution by team
record = df.groupby("TEAM_ABBREVIATION")["WL"].value_counts().unstack(fill_value=0)
record["win_pct"] = record["W"] / (record["W"] + record["L"])
record.sort_values("win_pct", ascending=False)

WL,L,W,win_pct
TEAM_ABBREVIATION,,,
BND,0,1,1.000000
TMC,0,2,1.000000
SHQ,0,2,1.000000
OKC,23,88,0.792793
CLE,26,69,0.726316
BOS,27,71,0.724490
HOU,35,58,0.623656
NYK,40,65,0.619048
IND,42,67,0.614679


## Play-by-Play Data

Fetch a single game's PBP to explore the output format.

In [21]:
from nba_api.stats.endpoints import playbyplayv3

# Pick a regular season game
sample_game_id = df.loc[df["GAME_TYPE"] == "Regular Season", "GAME_ID"].iloc[0]
print(f"Game ID: {sample_game_id}")

pbp = playbyplayv3.PlayByPlayV3(game_id=sample_game_id, start_period=1, end_period=10)
pbp_df = pbp.get_data_frames()[0]
print(f"{len(pbp_df)} plays, {pbp_df.shape[1]} columns")
print(f"\nColumns: {pbp_df.columns.tolist()}")
pbp_df.head(10)

Game ID: 0022401195
465 plays, 24 columns

Columns: ['gameId', 'actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId']


,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,...,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022401195,2,PT12M00.00S,1,0,,0,,,0,...,0,0,0,,Start of 1st Period (3:41 PM EST),period,start,0,0,1
1,0022401195,4,PT12M00.00S,1,1610612750,MIN,203497,Gobert,R. Gobert,0,...,,,0,h,Jump Ball Gobert vs. Filipowski: Tip to Juzang,Jump Ball,,1,0,2
2,0022401195,7,PT11M38.00S,1,1610612762,UTA,1642271,Filipowski,K. Filipowski,16,...,0,3,3,v,Filipowski 25' 3PT Step Back Jump Shot (3 PTS)...,Made Shot,Step Back Jump shot,1,3,3
3,0022401195,9,PT11M17.00S,1,1610612750,MIN,1630162,Edwards,A. Edwards,104,...,,,0,h,MISS Edwards 25' 3PT Pullup Jump Shot,Missed Shot,Pullup Jump shot,1,3,4
4,0022401195,10,PT11M14.00S,1,1610612762,UTA,1641729,Sensabaugh,B. Sensabaugh,0,...,,,0,v,Sensabaugh REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,5
5,0022401195,11,PT10M54.00S,1,1610612762,UTA,1641718,George,K. George,-188,...,,,0,v,MISS George 24' 3PT Step Back Jump Shot,Missed Shot,Step Back Jump shot,1,3,6
6,0022401195,12,PT10M50.00S,1,1610612750,MIN,203944,Randle,J. Randle,0,...,,,0,h,Randle REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,7
7,0022401195,13,PT10M45.00S,1,1610612750,MIN,1630183,McDaniels,J. McDaniels,-1,...,2,3,5,h,McDaniels Driving Layup (2 PTS) (Conley 1 AST),Made Shot,Driving Layup Shot,1,2,8
8,0022401195,15,PT10M36.00S,1,1610612762,UTA,1630548,Juzang,J. Juzang,-1,...,2,5,7,v,Juzang 1' Cutting Layup Shot (2 PTS) (Sensabau...,Made Shot,Cutting Layup Shot,1,2,9
9,0022401195,17,PT10M24.00S,1,1610612750,MIN,201144,Conley,M. Conley,-89,...,,,0,h,MISS Conley 26' 3PT Pullup Jump Shot,Missed Shot,Pullup Jump shot,1,3,10


In [ ]:
# Explore action types and structure
print("Action types:")
print(pbp_df["actionType"].value_counts())
print(f"\nDtypes:\n{pbp_df.dtypes}")

In [ ]:
from nba_api.stats.endpoints import playbyplayv3

# Pick a regular season game (GAME_ID starting with "002")
sample_game_id = df.loc[df["GAME_ID"].str.startswith("002"), "GAME_ID"].iloc[0]
print(f"Game ID: {sample_game_id}")

pbp = playbyplayv3.PlayByPlayV3(game_id=sample_game_id, start_period=1, end_period=10)
pbp_df = pbp.get_data_frames()[0]
print(f"{len(pbp_df)} plays, {pbp_df.shape[1]} columns")
print(f"\nColumns: {pbp_df.columns.tolist()}")
pbp_df.head(10)